# 00 · rawdata → seegdata

这一页只负责把 Neuracle 原始目录转换成 EEGLAB 的 `.set/.fdt`。转换函数是 `../matlab/raw_to_seegdata.m`，映射表是 `../metadata/raw_to_seeg_task_map.csv`。

默认开关全部关闭：不会重新导入，也不会覆盖现有 `seegdata`。test005 Task 2 的两个连续 raw session 会按映射顺序合并。

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess

PROJECT_ROOT = Path(r'E:/liulab_project/Project_colorieeg_2026')
MODULE_ROOT = PROJECT_ROOT / 'color_analyse_0727'
MAPPING_PATH = MODULE_ROOT / 'metadata' / 'raw_to_seeg_task_map.csv'
MATLAB_EXE = Path(r'E:/software/matlab_2021b/bin/matlab.exe')
EEGLAB_ROOT = Path(r'E:/matlab_tools/eeglab2025.1.0')
NEURACLE_PLUGIN_ROOT = EEGLAB_ROOT / 'plugins' / 'NeuracleEEGFileReader1.1.1'

mapping = pd.read_csv(MAPPING_PATH)
mapping

## 先做映射检查

每一行对应一个最终 `erpN` 文件；`raw_sessions` 中的分号表示同一个任务由多个连续 session 组成。

In [ ]:
def raw_session_path(subject, session):
    return PROJECT_ROOT / 'rawdata' / subject / session / '1' / '1'

checks = []
for row in mapping.itertuples(index=False):
    sessions = [item.strip() for item in row.raw_sessions.split(';')]
    for session in sessions:
        folder = raw_session_path(row.subject, session)
        checks.append({
            'subject': row.subject,
            'task_num': row.task_num,
            'session': session,
            'data_bdf': (folder / 'data.bdf').exists(),
            'evt_bdf': (folder / 'evt.bdf').exists() or (folder.parent / 'evt.bdf').exists(),
        })
checks = pd.DataFrame(checks)
checks['ready'] = checks['data_bdf'] & checks['evt_bdf']
checks

## 可选：执行转换

只有确认 `mapping` 和 `checks` 后，才把 `RUN_CONVERSION` 改成 `True`。`overwrite=False` 会保护已经生成的 `.set/.fdt`。如果只想重新检查路径，把 `RUN_DRY_RUN` 改成 `True`。

In [ ]:
RUN_CONVERSION = False
RUN_DRY_RUN = False
OVERWRITE = False

if RUN_CONVERSION or RUN_DRY_RUN:
    matlab_dir = (MODULE_ROOT / 'matlab').as_posix()
    matlab_code = (
        f"addpath('{matlab_dir}'); "
        f"raw_to_seegdata('{PROJECT_ROOT.as_posix()}',"
        f"'{EEGLAB_ROOT.as_posix()}',"
        f"'{NEURACLE_PLUGIN_ROOT.as_posix()}',"
        f"'{MAPPING_PATH.as_posix()}',"
        f"'overwrite',{str(OVERWRITE).lower()},'dry_run',{str(RUN_DRY_RUN).lower()});"
    )
    subprocess.run([str(MATLAB_EXE), '-batch', matlab_code], check=True)
else:
    print('Conversion disabled. Existing seegdata will be used.')

In [ ]:
outputs = []
for row in mapping.itertuples(index=False):
    output = PROJECT_ROOT / 'seegdata' / row.output_dir / f'{row.output_stem}.set'
    outputs.append({'subject': row.subject, 'task_num': row.task_num, 'set_exists': output.exists(), 'path': output})
pd.DataFrame(outputs)